In `features.py`, **`RollingSameVolCount`** is a **Data Integrity Metric**. It is designed to detect "Frozen Volume" or low-quality data feeds where the reported volume is duplicated across multiple days.

### 1. The Logic
In a healthy, active market, it is statistically nearly impossible for a stock to trade the *exact same number of shares* two days in a row (e.g., exactly 1,245,672 shares on Monday and exactly 1,245,672 shares on Tuesday). When this happens frequently, it usually indicates:
*   A "frozen" data feed.
*   The data provider is filling in missing values with the previous day's data.
*   A extremely illiquid stock where "0" or a placeholder is being reported.

### 2. The Mathematical Formula
1.  **Identify Duplicates:** Check if today's volume is exactly equal to yesterday's volume.
    $$\text{is\_duplicate}_t = (\text{Volume}_t == \text{Volume}_{t-1})$$
2.  **Rolling Sum:** Count how many times this has occurred over the rolling window.
    $$\text{RollingSameVolCount} = \sum_{t-window}^{t} \text{is\_duplicate}$$

### 3. Implementation Logic
**Conceptual Implementation:**
```python
# 1. Identify where volume exactly matches previous day
volume_diff = df_ohlcv["Volume"].groupby(level="Ticker").diff()
is_frozen = (volume_diff == 0)

# 2. Count occurrences over the quality window
rolling_same_vol_count = (
    is_frozen.groupby(level="Ticker")
    .rolling(window=quality_window)
    .sum()
)
```

### 4. Key Parameters (from `GLOBAL_SETTINGS`)
*   **Window (`quality_window`):** 252 days (1 trading year).
*   **Threshold (`max_same_vol_count`):** Set to **10**. 

### 5. Why it is used
If a stock matches its own volume more than 10 times in a year, it is flagged as **"Frozen Volume."** Even if the price data looks good, the volume data is untrustworthy. Since many indicators (like OBV, ATR, and Liquidity filters) rely on volume, this metric ensures that the "garbage in, garbage out" principle is applied—if the volume data is fake, the stock is excluded from the universe.

---  

In `features.py`, **`RollMedDollarVol`** stands for **Rolling Median Dollar Volume**. It is a liquidity metric used to ensure that a stock has enough trading volume to be safely traded without causing excessive market impact (slippage).

### 1. The Mathematical Formula
1.  **Daily Dollar Volume:** First, calculate the total dollar value of all shares traded on a single day.
    $$\text{Dollar Volume}_t = \text{Adjusted Close}_t \times \text{Volume}_t$$
2.  **Rolling Median:** Calculate the median of these daily values over a long window.
    $$\text{RollMedDollarVol} = \text{Median}(\text{Dollar Volume}_{t \dots t-252})$$

### 2. Implementation Details
The engine uses the **Median** rather than the **Mean** (Average) because volume often has massive outliers (e.g., a huge block trade or a one-day news event). The median provides a more accurate representation of the "typical" daily liquidity.

**Conceptual Implementation:**
```python
# 1. Calculate daily dollar turnover
dollar_vol = df_ohlcv["Adj Close"] * df_ohlcv["Volume"]

# 2. Calculate rolling median over the quality window
roll_med_dollar_vol = (
    dollar_vol.groupby(level="Ticker")
    .rolling(window=quality_window, min_periods=quality_min_periods)
    .median()
    .reset_index(level=0, drop=True)
)
```

### 3. Key Parameters (from `GLOBAL_SETTINGS`)
*   **Window (`quality_window`):** 252 days (1 trading year).
*   **Min Periods:** 126 days (ensures the stock has at least 6 months of data before it is "trusted").
*   **Threshold (`min_median_dollar_volume`):** Often set to **$1,000,000**. Stocks below this typical daily turnover are filtered out.

### 4. Why it is used
This metric is the primary **Liquidity Guardrail**.
*   It filters out "penny stocks" or "illiquid small-caps" that might show high returns but are impossible to buy or sell in large quantities in the real world.
*   By using a 1-year median, it prevents the system from being "tricked" by a stock that had one busy day but is normally dead.

---  

In `features.py`, **`RollingStalePct`** is a **Data Quality Metric**. It measures the percentage of days in a rolling window that a stock is considered "stale" (inactive or illiquid). This is used to filter out stocks with poor data quality or low liquidity.

### 1. The Definition of "Stale"
A stock is marked as "stale" on a specific day if either of the following is true:
1.  **Zero Volume:** No shares were traded (`Volume == 0`).
2.  **Zero Price Movement:** The high price is exactly equal to the low price (`High == Low`), indicating no trading activity occurred during the session.

### 2. The Mathematical Formula
$$\text{RollingStalePct} = \frac{\text{Number of Stale Days in Window}}{\text{Total Days in Window}}$$

### 3. Implementation Logic
While the raw calculation is often used during the feature generation phase, it is primarily driven by the thresholds in `GLOBAL_SETTINGS`.

**Conceptual Implementation:**
```python
# 1. Create a binary flag for "Stale"
is_stale = (df["Volume"] == 0) | (df["Adj High"] == df["Adj Low"])

# 2. Calculate the rolling percentage over the window
rolling_stale_pct = is_stale.rolling(window=quality_window, min_periods=quality_min_periods).mean()
```

### 4. Key Parameters (from `GLOBAL_SETTINGS`)
*   **Window (`quality_window`, `quality_min_periods`):** Typically **252 days, 126 days**.
*   **Threshold (`max_stale_pct`):** Often set to **0.05 (5%)**. 

### 5. Why it is used
If a stock has a high `RollingStalePct`, it is usually a "zombie stock" or has data feed issues. By including this as a feature, the model or the "Auditor" can automatically reject stocks that don't have enough active trading data to produce reliable technical signals. For example, an RSI or Beta calculation is mathematically meaningless if the stock hasn't traded for 10 out of the last 20 days.  

---  

In `features.py`, **`Ret_1d`** represents the **1-day percentage return** of a stock. It is the core input for many other technical indicators.

### 1. The Mathematical Formula
It is a standard percentage change calculation:
$$\text{Ret\_1d} = \frac{\text{Price}_t - \text{Price}_{t-1}}{\text{Price}_{t-1}}$$

### 2. Implementation Details
The calculation is handled by `QuantUtils.compute_returns` and orchestrated by `TickerEngine` to ensure it is calculated correctly per ticker:

**In `features.py`:**
```python
# The Orchestrator applies the kernel to each ticker's Adj Close series
rets = TickerEngine.map_kernels(df_ohlcv["Adj Close"], QuantUtils.compute_returns)
```

**In `core/quant.py`:**
```python
@staticmethod
def compute_returns(data):
    # Calculates day-over-day pct change and handles infinity errors
    return data.pct_change().replace([np.inf, -np.inf], np.nan)
```

### 3. Key Details
*   **Price Input:** Uses **Adjusted Close** to account for dividends and stock splits.
*   **Handling Infinity:** The code explicitly replaces `inf` and `-inf` with `NaN` to prevent downstream mathematical errors (e.g., if a stock price were to jump from $0.00 to a positive value).
*   **Alignment:** Because it uses `TickerEngine.map_kernels`, the first day of every ticker's history will be `NaN` (since there is no $t-1$ price), which is typically handled by `.fillna(0.0)` in later stages.
---  

In `features.py`, **`TRP`** stands for **True Range Percentage**. While `ATRP` provides a smoothed 14-day average of volatility, `TRP` represents the **raw, unsmoothed volatility** for a single day, expressed as a percentage of the price.

### 1. The Mathematical Formula
The calculation happens in two steps:

**Step A: Calculate Raw True Range (TR)**
The True Range accounts for the "gap" between today's trading range and yesterday's closing price:
$$\text{TR} = \max(\text{High} - \text{Low}, |\text{High} - \text{Close}_{\text{prev}}|, |\text{Low} - \text{Close}_{\text{prev}}|)$$

**Step B: Normalize by Price (TRP)**
The raw TR is then divided by the current adjusted close price:
$$\text{TRP} = \frac{\text{TR}}{\text{Current Price}}$$

### 2. Implementation Details
The code handles this by first calculating the raw TR kernel and then performing the division in the main feature assembly:

**In `core/quant.py`:**
```python
@staticmethod
def calculate_tr(high, low, close):
    prev_close = close.shift(1)
    # Vectorized max of (H-L, |H-Cp|, |L-Cp|)
    tr = np.maximum(
        high - low, 
        np.maximum((high - prev_close).abs(), (low - prev_close).abs())
    )
    return tr
```

**In `features.py`:**
```python
# From the vol_bundle calculation
trp = (vol_bundle["TR_Raw"] / df_ohlcv["Adj Close"]).fillna(0)
```

### 3. Interpretation
*   **TRP vs ATRP:** `TRP` is "noisy" because it only looks at one day. `ATRP` is "smooth" because it averages the last 14 days of TRP.
*   **High TRP:** Indicates a day with a very large price swing or a significant opening gap.
*   **Low TRP:** Indicates a "quiet" day where the stock traded in a very narrow range.

### 4. Why it is used
`TRP` is often used to identify **volatility spikes** or "outlier" days that might be masked by the smoothing of the 14-day ATR. In many strategies in this repository, `TRP` acts as a "real-time" risk measure to detect sudden price shocks.

---  

In `features.py`, **`ATRP`** stands for **Average True Range Percentage**. It is a normalized volatility indicator that expresses the average daily trading range as a percentage of the stock's price, allowing for direct volatility comparisons between different stocks.

### 1. The Mathematical Formula
The calculation happens in three stages:

**Stage A: True Range (TR)**
First, the raw daily range is calculated, accounting for gaps from the previous day's close:
$$\text{TR} = \max(\text{High} - \text{Low}, |\text{High} - \text{Close}_{\text{prev}}|, |\text{Low} - \text{Close}_{\text{prev}}|)$$

**Stage B: Average True Range (ATR)**
The TR is smoothed using **Wilder’s Smoothing** (an Exponential Moving Average where $\alpha = 1/\text{period}$):
$$\text{ATR}_t = \text{ATR}_{t-1} + \frac{\text{TR}_t - \text{ATR}_{t-1}}{\text{period}}$$

**Stage C: Normalization (ATRP)**
Finally, the ATR is divided by the current price to get the percentage:
$$\text{ATRP} = \frac{\text{ATR}}{\text{Current Price}}$$

### 2. Implementation Details
The code in `features.py` and `core/quant.py` implements it as follows:

**In `core/quant.py`:**
```python
# Stage A & B
def calculate_atr(high, low, close, period):
    tr = calculate_tr(high, low, close)
    return tr.ewm(alpha=1 / period, adjust=False).mean()
```

**In `features.py`:**
```python
# Stage C
atr = vol_bundle["ATR_Smooth"]
natr = (atr / df_ohlcv["Adj Close"]).fillna(0) # This is ATRP
```

### 3. Interpretation
*   **ATRP = 0.02:** This means the stock typically moves about 2% per day.
*   **High ATRP:** Indicates a highly volatile stock (e.g., a small-cap biotech or a leveraged ETF).
*   **Low ATRP:** Indicates a stable, low-volatility stock (e.g., a large-cap utility or a treasury bond ETF).

### 4. Key Parameters
*   **Period:** Defaulted to **14 days** (`atr_period` in `GLOBAL_SETTINGS`).
*   **Normalization:** By dividing by the current price, `ATRP` remains stable even if a stock's price doubles, whereas raw `ATR` would also double.

---

In `features.py`, **`Mom_21`** (Momentum 21) measures the percentage price change of a stock over the last 21 trading days (approximately one trading month).

### 1. The Mathematical Formula
It is a simple point-to-point percentage return calculation:
$$\text{Mom\_21} = \frac{\text{Price}_t - \text{Price}_{t-21}}{\text{Price}_{t-21}}$$

### 2. Implementation in `features.py`
The calculation is performed using the pandas `pct_change` method on the grouped adjusted close prices:

```python
# From features.py
mom_21 = grouped["Adj Close"].pct_change(win_21d)
```

### 3. Interpretation
*   **Positive Value (e.g., 0.10):** The stock has risen 10% over the last 21 trading days.
*   **Negative Value (e.g., -0.05):** The stock has fallen 5% over the last 21 trading days.
*   **Zero:** The price is exactly the same as it was 21 days ago.

### 4. Key Details
*   **Window Size (`win_21d`):** Defined as **21 days** in `GLOBAL_SETTINGS`. This is the standard "Monthly Momentum" anchor used in many quantitative strategies.
*   **Price Input:** Uses **Adjusted Close** to ensure that dividends and stock splits do not create "fake" momentum or price drops.
*   **Relationship with IR_63:** While `Mom_21` looks at raw returns over 1 month, `IR_63` looks at risk-adjusted returns over 3 months. Together, they help the model distinguish between a sudden spike and a sustained trend.
---   

In `features.py`, **`Consistency`** measures the reliability of a stock's upward movement. Instead of looking at *how much* the price moved (magnitude), it looks at *how often* the price moved higher (frequency) over a specific timeframe.

### 1. The Mathematical Formula
It calculates the percentage of days that had a positive return over a rolling 5-day window:
$$\text{Consistency} = \frac{\text{Number of Positive Days in Window}}{\text{Total Days in Window (5)}}$$

### 2. Implementation in `features.py`
The calculation is performed using a boolean check on daily returns, followed by a rolling mean:

```python
consistency = (
    (rets > 0)                                 # 1. Create a series of True/False (1 or 0)
    .astype(float)                             # 2. Convert True/False to 1.0/0.0
    .groupby(level="Ticker")                   # 3. Group by Ticker
    .rolling(win_5d)                           # 4. Use 5-day window
    .mean()                                    # 5. Calculate average (percentage of 1s)
    .reset_index(level=0, drop=True)           # 6. Align with original index
)
```

### 3. Interpretation
The value ranges from **0.0** to **1.0**:
*   **1.0:** The stock went up **5 out of 5 days** (Perfect consistency).
*   **0.8:** The stock went up 4 out of 5 days.
*   **0.6:** The stock went up 3 out of 5 days.
*   **0.0:** The stock went down (or was flat) every day for the last 5 days.

### 4. Why it is used
This metric is designed to filter out "noisy" or "erratic" stocks. 
*   **High Consistency:** Indicates a "grinder" — a stock that moves up steadily and predictably.
*   **Low Consistency with High Returns:** Indicates a "gapper" — a stock that might have had one massive 20% day but was actually falling the other 4 days. 

In many of the strategies in this repo, a high consistency score is a prerequisite for entering a momentum trade.

### 5. Key Parameters
*   **Window (`win_5d`):** 5 trading days (one week).
*   **Logic:** Only counts strictly positive returns (`rets > 0`). Flat days (0.0% change) are treated as non-positive.

---  


In `features.py`, **`Convexity`** measures the **acceleration** or **deceleration** of a stock's price trend. It identifies if a trend is gaining strength (convex) or losing momentum and "rounding over" (concave).

### 1. The Mathematical Formula
Convexity is calculated as the **2-day difference of the 5-day Price Slope**:
$$\text{Convexity}_t = \text{Slope\_P\_5}_t - \text{Slope\_P\_5}_{t-2}$$

### 2. Implementation Details
The calculation is performed in two steps within `features.py`:

**Step A: Calculate Price Slope**
First, the engine calculates the log-price slope over 5 days (`Slope_P_5`).

**Step B: Calculate Change in Slope**
The `calculate_convexity_5d_fast` kernel is then applied to that slope series:
```python
# In core/quant.py
@staticmethod
def calculate_convexity_5d_fast(slope_series: pd.Series) -> pd.Series:
    """Measures change in slope (Acceleration)."""
    return slope_series.diff(2).fillna(0)
```

### 3. Interpretation (The "Physics" of Price)
If we think of price as "position":
*   **Price:** Position.
*   **Slope_P_5:** Velocity (how fast the price is moving).
*   **Convexity:** Acceleration (how fast the velocity is changing).

| Value | Meaning | Market Context |
| :--- | :--- | :--- |
| **High Positive** | Accelerating Uptrend | "Parabolic" move; strong buying pressure is intensifying. |
| **Zero** | Constant Trend | Price is moving at a steady, predictable rate. |
| **High Negative** | Decelerating / Exhaustion | The trend is "rolling over." Even if the price is still rising, it is doing so more slowly, often signaling a peak. |

### 4. Strategy Usage
In this codebase, a `convexity_exit` threshold (defined as **-0.7** in `GLOBAL_SETTINGS`) is often used to exit trades before a full reversal happens, catching the moment the "upward velocity" begins to collapse.

---  

In `features.py`, **`Slope_V_5`** measures the 5-day rolling trend of the stock's **On-Balance Volume (OBV)**. Like the price slope, it uses a vectorized 5-day OLS linear regression to determine if volume pressure is increasing or decreasing.

### 1. The Mathematical Formula
It applies the same 5-day OLS simplified slope formula used for price, but applies it to the OBV series:
$$\text{Slope\_V} = \frac{2 \cdot \text{OBV}_t + 1 \cdot \text{OBV}_{t-1} + 0 \cdot \text{OBV}_{t-2} - 1 \cdot \text{OBV}_{t-3} - 2 \cdot \text{OBV}_{t-4}}{10}$$

### 2. Implementation Logic
The calculation involves two distinct steps to ensure accuracy across different stocks:

#### Step A: Normalized OBV Calculation
To make volume trends comparable between a "mega-cap" stock (millions of shares) and a "small-cap" stock (thousands of shares), the volume is normalized by its 63-day rolling mean before calculating OBV:
```python
# From features.py: get_obv_kernel
v_baseline = volume.rolling(window=63, min_periods=1).mean()
v_rel = volume / v_baseline
obv = cumulative_sum(sign(price_change) * v_rel)


#### Step B: Slope Calculation
The 5-day slope kernel is then applied to this normalized OBV:
```python
slope_v = TickerEngine.map_kernels(obv, QuantUtils.calculate_rolling_slope_5d_fast)
```

### 3. Interpretation
This metric captures **Volume Momentum**:
*   **Positive Value:** Indicates "Volume Accumulation." Buying pressure (on up days) is outweighing selling pressure, and this trend is accelerating/persisting over the last 5 days.
*   **Negative Value:** Indicates "Volume Distribution." Selling pressure is dominant.
*   **Divergence:** If `Slope_P_5` (Price) is positive but `Slope_V_5` (Volume) is negative, it may suggest a weakening trend (price is rising on decreasing relative volume).

### 4. Key Parameters
*   **Window:** 5 trading days (one week).
*   **Normalization:** 63-day relative volume baseline.
*   **Kernel:** Uses the same weighted OLS sum as `Slope_P_5`.

---  

✦ In features.py, Slope_P_5 measures the 5-day rolling growth rate of the stock's price using a vectorized Ordinary Least  Squares (OLS) linear regression.

  1. The Mathematical Formula
  It calculates the slope of the best-fit line over the last 5 days. For a 5-day window ($n=5$), the OLS slope formula
  simplifies to a weighted sum of the prices:
  $$\text{Slope} = \frac{2P_t + 1P_{t-1} + 0P_{t-2} - 1P_{t-3} - 2P_{t-4}}{10}$$

  2. Key Implementation Detail: Log Prices
  In features.py, the slope is calculated on the logarithm of the price:

   1 log_price = np.log(df_ohlcv["Adj Close"].replace(0, 1e-8))
   2 slope_p = TickerEngine.map_kernels(
   3     log_price, QuantUtils.calculate_rolling_slope_5d_fast
   4 )
  Because it uses log prices, the resulting slope represents the continuous growth rate (or log-return momentum) rather
  than a raw dollar-change slope. This makes the values comparable across stocks with different price levels.

  3. Code Breakdown
  The "fast" calculation in core/quant.py avoids heavy regression libraries by using the simplified weights:

    1 @staticmethod
    2 def calculate_rolling_slope_5d_fast(series):
    3     # n=5 weighted sum derivation:
    4     return (
    5         2 * series              # Today
    6         + 1 * series.shift(1)   # Yesterday
    7         + 0 * series.shift(2)   # ...
    8         + -1 * series.shift(3)
    9         + -2 * series.shift(4)
   10     ) / 10.0

  4. Interpretation
   * Positive Value: The stock is in an uptrend over the last 5 days.
   * Negative Value: The stock is in a downtrend over the last 5 days.
   * Magnitude: Since it is based on log prices, a value of 0.01 roughly corresponds to a 1% average daily growth rate
     over the window.
---  

In `features.py`, **`Range_Pos_20`** (Range Position 20) is a stochastic-style indicator that measures where the current price sits relative to its high-low range over the last 20 trading days.

### 1. The Mathematical Formula
$$\text{Range\_Pos\_20} = \frac{\text{Current Close} - \text{Rolling Low}}{\text{Rolling High} - \text{Rolling Low}}$$

### 2. Interpretation
The value is bounded (conceptually) between **0.0** and **1.0**:
*   **1.0:** The current price is at the highest high of the last 20 days.
*   **0.0:** The current price is at the lowest low of the last 20 days.
*   **0.5:** The current price is exactly in the middle of the 20-day range.

### 3. Implementation Details
The calculation is handled by `QuantUtils.calculate_range_pos` in `core/quant.py`:

```python
@staticmethod
def calculate_range_pos(high, low, close, window=20):
    # 1. Identify the 20-day rolling high and rolling low
    roll_min = low.rolling(window=window).min()
    roll_max = high.rolling(window=window).max()

    # 2. Prevent division by zero if the range is flat
    denom = (roll_max - roll_min).replace(0, 1e-8)

    # 3. Calculate position within that range
    return (close - roll_min) / denom
```

### 4. Key Parameters
*   **`window=20`**: Uses a 20-day lookback, representing approximately one trading month.
*   **Inputs**: It uses the `Adj High`, `Adj Low`, and `Adj Close` prices.
*   **Protection**: It includes a replacement for zero denominators (`1e-8`) to prevent mathematical errors in stocks with no price movement.

---  



In `features.py`, **`AutoCorr_15`** measures the serial correlation (autocorrelation) of a stock's daily returns over a rolling 15-day window. It is used to identify "memory" or persistence in price movements.

### 1. The Mathematical Formula
It calculates the Pearson correlation coefficient between the current day's return ($R_t$) and the previous day's return ($R_{t-1}$) over the specified window:
$$\text{AutoCorr} = \text{Correlation}(R_t, R_{t-1}) \text{ over 15 days}$$

### 2. Interpretation
*   **Positive Value (> 0):** Indicates **Trending/Persistence**. If the stock went up yesterday, it is likely to go up today.
*   **Negative Value (< 0):** Indicates **Mean Reverting/Anti-persistence**. If the stock went up yesterday, it is likely to go down today.
*   **Near Zero:** Indicates the returns behave like a "Random Walk" with no significant memory.

### 3. Implementation Details
The calculation is handled by `QuantUtils.calculate_autocorr` in `core/quant.py` and applied in `features.py`:

**In `features.py`:**
```python
autocorr_15 = TickerEngine.map_kernels(
    rets, QuantUtils.calculate_autocorr, lag=1, window=15
)
```

**In `core/quant.py`:**
```python
@staticmethod
def calculate_autocorr(rets, lag=1, window=15):
    # Vectorized: Correlation of returns with their previous self (shifted by lag)
    return rets.rolling(window=window).corr(rets.shift(lag)).fillna(0.0)
```

### 4. Key Parameters
*   **`window=15`**: A short-term 15-trading day lookback (~3 weeks).
*   **`lag=1`**: Compares today's return to specifically yesterday's return.
*   **`fillna(0.0)`**: Any period with insufficient data (like the first 15 days of a ticker's history) is defaulted to 0.0.

---  

In `features.py`, **`DD_21`** represents the rolling 21-day drawdown from the peak price. It measures how much the current price has dropped relative to its highest point over the last 21 trading days (approximately one month).

### 1. The Mathematical Formula
$$\text{DD\_21} = \left( \frac{\text{Current Price}}{\text{Maximum Price over 21 days}} \right) - 1$$

*   A value of `0.0` means the stock is currently at its 21-day high.
*   A value of `-0.05` means the stock is 5% below its 21-day high.

### 2. Implementation in `features.py`
The calculation is performed using a rolling maximum of the `Adj Close` price:

```python
dd_21 = (
    df_ohlcv["Adj Close"]
    / grouped["Adj Close"].rolling(win_21d).max().reset_index(level=0, drop=True)
) - 1.0
```

### 3. Key Details
*   **Window Size:** It uses `win_21d`, which is defined as **21 days** in `GLOBAL_SETTINGS`.
*   **Price Input:** It uses the **Adjusted Close** price, which accounts for corporate actions like dividends and splits.
*   **Handling NaNs:** In the final assembly of features, any initial `NaN` values (from the start of the window) are filled with `0` using `.fillna(0)`.
*   **Significance:** In the strategy registry (`strategy/registry.py`), this metric is often used as a **"Dip Buyer"** indicator (calculated as `-DD_21`), where a more negative drawdown results in a higher "dip buying" score.

---  

In `features.py`, **`Beta_63`** represents the 63-day rolling beta of a stock relative to the market benchmark (typically SPY). It is calculated through the following process:

### 1. The Mathematical Formula
The calculation follows the standard financial formula for Beta:
$$\text{Beta} = \frac{\text{Covariance}(R_{\text{stock}}, R_{\text{benchmark}})}{\text{Variance}(R_{\text{benchmark}})}$$

### 2. Implementation Details
The calculation is performed in `core/features.py` using a unified pattern that delegates the math to `core/quant.py`:

*   **Window Size:** It uses a 63-day rolling window (`win_63d`), which corresponds to approximately one fiscal quarter of trading data.
*   **Returns:** It uses daily percentage changes (`rets`) for both the individual stock and the market benchmark (`mkt_ret_series`).
*   **Orchestration:** The `TickerEngine.map_kernels` function ensures this calculation is applied ticker-by-ticker across the entire dataset.

### 3. Code Breakdown
In `core/quant.py`, the `calculate_rolling_beta` method performs the core logic:

```python
@staticmethod
def calculate_rolling_beta(rets, benchmark_rets, window):
    # 1. Calculate rolling covariance between stock and benchmark
    cov = rets.rolling(window).cov(benchmark_rets)
    
    # 2. Calculate rolling variance of the benchmark
    var = benchmark_rets.rolling(window).var()

    # 3. Divide Cov/Var and fill initial NaN values with 1.0 (neutral beta)
    return (cov / var).fillna(1.0)
```

### 4. Data Inputs
*   **`rets`**: Daily adjusted close returns for the specific ticker.
*   **`benchmark_rets`**: Daily returns of the benchmark ticker defined in `GLOBAL_SETTINGS["benchmark_ticker"]` (defaulting to **SPY**).
*   **`window`**: Defaulted to **63** via `GLOBAL_SETTINGS["63d_window"]`.

---  

✦ The RSI (Relative Strength Index) in this project is calculated using Wilder’s Smoothing, which is the industry-standard method used by platforms
  like TradingView.

  The logic is split into two parts: the mathematical kernel in core/quant.py and the orchestration in core/features.py.

  1. The Mathematical Kernel (core/quant.py)
  The calculate_rsi method follows these steps:

   1. Calculate Change: It takes the daily price difference (delta = series.diff()).
   2. Separate Gains and Losses:
       * up: All positive changes (negative changes become 0).
       * down: All negative changes as positive values (positive changes become 0).
   3. Smoothing (The Wilder's Way): It uses an Exponential Weighted Moving Average (ewm) with an alpha of 1 / period.
       * ma_up = up.ewm(alpha=1/period).mean()
       * ma_down = down.ewm(alpha=1/period).mean()
   4. Calculate RS: rs = ma_up / ma_down
   5. Final Formula: rsi = 100 - (100 / (1 + rs))
   6. Cleanup:
       * If ma_down is 0, RSI becomes 100 (Pure upward momentum).
       * Missing values at the start of the series are filled with 50 (Neutral).

  2. Implementation Context (core/features.py)
  In the feature generation layer:
   * The system uses TickerEngine.map_kernels to ensure the RSI is calculated independently for each ticker (preventing data from one stock's history
     from bleeding into another).
   * The standard period is set via GLOBAL_SETTINGS["rsi_period"] (defaulting to 14).

  3. Usage in Strategy (strategy/registry.py)
  In the strategy registry, the metric is named "Oversold (-RSI)".
   * It is inverted (-obs.rsi) so that a higher number represents a more "Oversold" state (higher pressure).
   * The signal triggers are centralized in settings: RSI 30 for Oversold and RSI 70 for Overbought.

---    

✦ The IR_63 (Information Ratio over 63 days) is a measure of Risk-Adjusted Alpha. It tells you how consistently a stock is beating the benchmark (SPY)
  relative to the "noise" (volatility) of its outperformance.

  Here is the breakdown of the calculation from core/features.py:

  1. The Logic
  The Information Ratio is mathematically defined as:
  $$IR = \frac{\text{Average Active Return}}{\text{Tracking Error (StdDev of Active Returns)}}$$

   1. Calculate Active Return: Subtract the benchmark return from the stock's return for each day ($R_{ticker} - R_{market}$).
   2. Rolling Mean: Calculate the average of those differences over the last 63 days.
   3. Rolling StdDev: Calculate the standard deviation of those differences over the same 63 days.
   4. The Ratio: Divide the Mean by the StdDev.

  2. The Code

    1 # 1. Calculate Active Returns (Stock Return - SPY Return)
    2 active_ret = rets.sub(mkt_ret_series, axis=0, level="Date")
    3
    4 # 2. Set up the 63-day rolling window per ticker
    5 roll_active = active_ret.groupby(level="Ticker").rolling(window=63)
    6
    7 # 3. Calculate Mean / StdDev
    8 ir_63 = (
    9     (roll_active.mean() / roll_active.std())
   10     .reset_index(level=0, drop=True)
   11     .fillna(0)
   12 )

  3. What it tells the Agent
  In the strategy registry (strategy/registry.py), this is referred to as the "Gatekeeper" for the Trend Engine:

   * High IR (> 0.5): The stock is beating the market steadily. This suggests a "Quality" trend driven by institutional accumulation.
   * Low/Negative IR: The stock is either underperforming or its outperformance is extremely erratic (high "Tracking Error"). The system treats this
     as a "Random Walk" and avoids it.

  4. Comparison to Sharpe Ratio
   * Sharpe Ratio: Measures return relative to Total Risk (volatility).
   * Information Ratio (IR): Measures return relative to Relative Risk (how much it deviates from the benchmark).

  Note: In the feature layer, the IR is calculated on a daily scale (it is not annualized by $\sqrt{252}$ like the Sharpe ratio often is in the
  performance reports).

---

The current implementation of `Slope_V_5` suffers from **"Magnitude Distortion,"** which causes the system to prioritize high-volume stocks (Large-Caps) regardless of their actual performance.

### The Problem: Absolute vs. Relative Pressure
The current formula uses the OLS Slope of **Raw OBV**. Since OBV is calculated using the absolute number of shares traded, the signal scale is tied to a stock's liquidity rather than its intensity.

1.  **Large-Cap Dominance:** A stock like AMD trading 50 million shares has a "Slope" that is mathematically 1,000x larger than a small-cap stock trading 50,000 shares. 
2.  **Cross-Sectional Washing:** When you calculate the Z-score across the entire market, the Large-Caps act as massive outliers. This "squashes" the small-caps' Z-scores toward zero, making it impossible for the system to detect "smart money" entering a smaller stock.
3.  **Price Blindness:** It treats 1 million shares of a $1 stock (garbage) exactly the same as 1 million shares of a $500 stock (institutional flow).

---

### The Fix: "Intensity Normalization"
To make the signal work across a diverse universe, the system needs to switch from **Absolute Volume** to **Relative Intensity**.

#### The Logical Adjustment:
Instead of taking the slope of `Cumulative(Sign * Raw_Volume)`, you calculate the slope of **Relative OBV**.

**The New Formula:**
1.  **Normalize Volume:** Divide today's volume by its 20-day moving average:
    $$\text{V}_{norm} = \frac{\text{Volume}_{t}}{\text{SMA}(\text{Volume}, 20)}$$
2.  **Directionalize:** Multiply by the sign of the price change:
    $$\text{V}_{intensity} = \text{Sign}(\Delta \text{Price}) \times \text{V}_{norm}$$
3.  **Slope:** Apply the 5-day OLS Slope to the cumulative sum of $\text{V}_{intensity}$.

### Summary of the Benefit
| Feature | Current `Slope_V_5` | Fixed `Slope_V_5` |
| :--- | :--- | :--- |
| **Focus** | How many **shares** moved? | How **unusual** is today's volume? |
| **Bias** | Favors Large-Caps / Penny Stocks. | Neutral across all market caps. |
| **Signal** | "Total Share Pressure." | "Conviction Intensity." |
| **Z-Score** | Skewed by high-volume outliers. | Evenly distributed and comparable. |

**In short:** The fix ensures that a stock trading **3x its normal volume** generates a strong signal, whether it is a $2 trillion tech giant or a $200 million niche company.

# Project Alpha: RL Bot v63p

## 📋 TODO & Roadmap
- **Data Policy:** Should we apply `handle_zeros_as_nan` globally to all matrices?
- **Feature Scaling:** Finalize the `get_agent_view` dual-view scaling logic.
- **Performance:** Move heavy logic (`analyze_4d_regime`) into `core` modular library.

<details>
<summary><b>📜 Version History (v59 - v63)</b></summary>

### v63
- Added **Metric Registry** (Strong-typed, dual-view).
- Fixed **Index Poisoning** bug (Dates in Ticker column).
- Unified **Perception** and **Execution** in Audit packs.
- Added 4 New Pillars: Return Autocorr, Range Position, OBV Divergence, Convexity.

### v62
- Fixed **Object Reference Mutation** bug in `compute_alpha_ensemble`.
- Added `PostBuildVerifier`, `DeepDiveDebugger`, and `ForensicExporter`.

### v61
- Verified all metrics with Excel parity.
- Refactored `AlphaEngine` into Orchestrator pattern.

### v60
- Converted notebook code to modular system.
- Added Volatility Regime plots.

### v59
- Implemented **Result Pattern** for error handling.
- Switched to logarithmic returns globally.
</details>

## I. Initialization & Environment

In [1]:
# 1. Setup & Path
%load_ext autoreload
%autoreload 2

import sys
import os
import gc
import pandas as pd
import numpy as np
import multiprocessing
import random
import re
import time
from pathlib import Path
from dataclasses import dataclass
from typing import Literal, Optional
from types import SimpleNamespace
from IPython.display import display, clear_output
from tqdm.notebook import tqdm
import plotly.io as pio

def add_project_root_to_path():
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            return path
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Could not find notebooks_RLVR directory")

add_project_root_to_path()

# 2. Display Settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.precision", 4)

# 3. Project Imports
from core.analyzer import create_walk_forward_analyzer, run_headless_simulation
from core.auditor import SystemAuditor as SA
from core.builder import ParallelFeatureBuilder, FeatureCubeStitcher
from core.contracts import FilterPack, EngineInput, MarketObservation
from core.engine import AlphaEngine, AlphaCache
from core.environment import DiscoveryEnv
from core.features import generate_features
from core.logic import AlphaLogic
from core.paths import OUTPUT_DIR
from core.quant import QuantUtils, TickerEngine
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import STRATEGY_REGISTRY

# 4. Load Environment Paths
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"✓ Paths Initialized.\n  OHLCV: {DATA_PATH_OHLCV}\n  Indices: {DATA_PATH_INDICES}")

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output

✓ Paths Initialized.
  OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
  Indices: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


## II. Data Pipeline

In [ ]:
print("⏳ Loading DataFrames... takes ~6 minutes")
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
)

print("⚡ Generating Features...")
features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    df_fed=df_fed,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

print("🚀 Unstacking Wide Matrices...")
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

In [ ]:
# === PERSISTENCE SANDBOX ===
features_df.to_parquet("output/features_df.parquet", index=True)
macro_df.to_parquet("output/macro_df.parquet", index=True)
df_close_wide.to_parquet("output/df_close_wide.parquet", index=True)
df_atrp_wide.to_parquet("output/df_atrp_wide.parquet", index=True)
df_trp_wide.to_parquet("output/df_trp_wide.parquet", index=True)

In [2]:
# === RELOADING FROM PARQUET ===
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")

features_df = pd.read_parquet("output/features_df.parquet")
macro_df = pd.read_parquet("output/macro_df.parquet")
df_close_wide = pd.read_parquet("output/df_close_wide.parquet")
df_atrp_wide = pd.read_parquet("output/df_atrp_wide.parquet")
df_trp_wide = pd.read_parquet("output/df_trp_wide.parquet")

In [3]:
import pandas as pd

# 1. Setup Filters
ticker_to_audit = "TSLA"
idx = pd.IndexSlice

# 2. Export Calculated Features (The "Answers")
subset_df = features_df.loc[idx[ticker_to_audit, :], :]

# Derive date range from the retrieved TSLA features
start_date = subset_df.index.get_level_values(1).min()
end_date = subset_df.index.get_level_values(1).max()

subset_df.to_csv(
    OUTPUT_DIR / f"{ticker_to_audit}_features_audit_2025.csv",
    index=True,
    encoding="utf-8-sig",
)

# 3. Export Raw Price/Volume (The "Inputs")
# We include 'SPY' for Beta/IR calculations
all_tickers = [ticker_to_audit, "SPY"]
raw_ohlcv = df_ohlcv.loc[idx[all_tickers, start_date:end_date], :]
raw_ohlcv.to_csv(OUTPUT_DIR / f"{ticker_to_audit}_raw_price_audit.csv", index=True)

# 4. Export Macro Data (The "Market Context")
# Contains Mkt_Ret and Macro_Trend needed for IR_63 and Beta_63
raw_macro = macro_df.loc[start_date:end_date, :]
raw_macro.to_csv(OUTPUT_DIR / f"{ticker_to_audit}_raw_macro_audit.csv", index=True)

print(
    "✅ All 3 audit files generated:\n"
    f"   1. {ticker_to_audit}_features_audit_2025.csv\n"
    f"   2. {ticker_to_audit}_raw_price_audit.csv\n"
    f"   3. {ticker_to_audit}_raw_macro_audit.csv\n"
    "\nYou can now replicate the features file using the raw files in Excel."
)

✅ All 3 audit files generated:
   1. TSLA_features_audit_2025.csv
   2. TSLA_raw_price_audit.csv
   3. TSLA_raw_macro_audit.csv

You can now replicate the features file using the raw files in Excel.


In [ ]:
# Option 2: Temporary context (cleanest — no side effects)
with pd.option_context("display.float_format", "{:.6f}".format):
    print(macro_df.loc[start_date:end_date])